# 01. Demand Data Exploration & Foundation

**Project:** Fodripio - Forecast-Driven Production Planning and Inventory Optimization Under Uncertain Demand  
**Phase:** 0 - Project Setup & Data Foundation  

## Overview
This notebook loads the raw and processed demand dataset generated by `SyntheticDemandGenerator` and ingested via `DataLoader` to explore overall demand distributions, seasonality, trends, and outlier flagging.

In [1]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import yaml

# Ensure project root is in python path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.data.synthetic import SyntheticDemandGenerator
from src.data.load import DataLoader

plt.style.use('seaborn-v0_8-whitegrid')
print("Libraries imported successfully.")

Libraries imported successfully.


## 1. Load Experiment Configuration

In [2]:
config_path = os.path.join(project_root, 'configs', 'experiment.yaml')
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

print("Experiment Config:", config['experiment'])
print("Cost Assumptions:", config['costs'])

Experiment Config: {'name': 'fodripio_base_experiment', 'seed': 42, 'horizon': 14}
Cost Assumptions: {'holding_cost_per_unit_day': 1.5, 'shortage_cost_per_unit_day': 15.0, 'setup_cost_per_batch': 200.0, 'overtime_cost_per_hour': 45.0}


## 2. Load Processed Demand Data

In [ ]:
processed_path = os.path.join(project_root, config['data']['processed_path'])
loader = DataLoader(processed_path)
df = loader.load_raw_data()
loader.process()
print("Data summary:", loader.summarize())
df.head()

2026-09-15 19:01:23,513 - INFO - Loading raw demand data from C:\zendvalmics\project\fodripio\data\processed\demand_processed.csv
2026-09-15 19:01:23,539 - INFO - Loaded 3655 rows spanning 2024-01-01 00:00:00 to 2025-12-31 00:00:00


,date,product_id,demand,price,promotion,holiday,shock,is_outlier
0,2024-01-01,P1,59.28,100.78,0,1,0,False
1,2024-01-01,P2,68.33,127.06,0,1,0,False
2,2024-01-01,P3,122.09,149.01,0,1,0,False
3,2024-01-01,P4,49.24,171.94,0,1,0,False
4,2024-01-01,P5,18.16,202.66,0,1,0,False


## 3. Product Demand Distributions

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
sns.boxplot(data=df, x='product_id', y='demand', palette='viridis', ax=ax)
ax.set_title("Daily Demand Distributions by Product", fontsize=14)
ax.set_xlabel("Product ID")
ax.set_ylabel("Daily Demand (Units)")
plt.tight_layout()
plt.show()

## 4. Time-Series Trajectory per Product

In [ ]:
fig, axes = plt.subplots(5, 1, figsize=(14, 12), sharex=True)
for idx, (p_id, p_df) in enumerate(df.groupby('product_id')):
    axes[idx].plot(p_df['date'], p_df['demand'], label=f'{p_id} Demand', color='tab:blue', alpha=0.7)
    outliers = p_df[p_df['is_outlier']]
    axes[idx].scatter(outliers['date'], outliers['demand'], color='red', s=20, label='Outlier Flag', zorder=5)
    axes[idx].set_ylabel(p_id)
    axes[idx].legend(loc='upper right')

axes[-1].set_xlabel("Date")
plt.suptitle("Daily Demand Time-Series and Outliers Across Products", fontsize=14, y=0.99)
plt.tight_layout()
plt.show()